## 1. Utilities

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import GroupKFold

DATA_PATH = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")

def recent_mean_diff(values, window):
    values = values[-(window + 1):]
    return float(np.diff(values).mean()) if len(values) >= 2 else 0.0

def recent_slope(y_values, x_values, window):
    y, x = y_values[-window:], x_values[-window:]
    if len(y) < 2: return 0.0
    centered_x = x - x.mean()
    denom = float(np.dot(centered_x, centered_x))
    return float(np.dot(centered_x, y - y.mean()) / denom) if denom != 0.0 else 0.0

def nearest_index(sorted_values, target):
    idx = int(np.searchsorted(sorted_values, target, side='left'))
    if idx >= len(sorted_values): return len(sorted_values) - 1
    if idx > 0 and abs(sorted_values[idx - 1] - target) <= abs(sorted_values[idx] - target):
        return idx - 1
    return idx

def beam_predict(gr_values, tw_tvt, tw_gr, start_tvt, beam_size, move_cost, emit_scale, radius=2):
    start_idx = nearest_index(tw_tvt, start_tvt)
    s = pd.Series(gr_values).interpolate(limit_direction='both').fillna(np.nanmean(tw_gr))
    smoothed_gr = s.rolling(radius * 2 + 1, center=True, min_periods=1).mean().values
    
    states = {start_idx: 0.0}
    backpointers = []
    for gr_value in smoothed_gr:
        candidates, parents = {}, {}
        for idx, cost in states.items():
            for delta in (-1, 0, 1):
                n_idx = idx + delta
                if 0 <= n_idx < len(tw_tvt):
                    emit_cost = ((gr_value - tw_gr[n_idx]) ** 2) / emit_scale
                    total_cost = cost + emit_cost + move_cost * abs(delta)
                    if n_idx not in candidates or total_cost < candidates[n_idx]:
                        candidates[n_idx], parents[n_idx] = total_cost, idx
        kept = sorted(candidates.items(), key=lambda x: x[1])[:beam_size]
        states = {idx: cost for idx, cost in kept}
        backpointers.append({idx: parents[idx] for idx, _ in kept})
    
    curr = min(states, key=states.get)
    path = [curr]
    for step in range(len(backpointers)-1, 0, -1):
        curr = backpointers[step][curr]
        path.append(curr)
    return tw_tvt[np.array(path[::-1], dtype=np.int32)]

## 2. Elite Feature Engineering (50 фіч)

In [ ]:
def build_hidden_features(horizontal_path, typewell_path, is_train):
    well = horizontal_path.name.split('__')[0]
    df = pd.read_csv(horizontal_path)
    mask = df['TVT_input'].isna().to_numpy()
    if not mask.any(): return None
    mask_start = int(np.flatnonzero(mask)[0])
    
    known, hidden = df.iloc[:mask_start].copy(), df.iloc[mask_start:].copy()
    tw = pd.read_csv(typewell_path)
    tw_tvt, tw_gr = tw['TVT'].values.astype(np.float32), tw['GR'].values.astype(np.float32)
    last_known = known.iloc[-1]
    last_known_tvt = float(last_known['TVT_input'])
    
    gr_full = df['GR'].interpolate(limit_direction='both').fillna(np.nanmean(tw_gr))
    
    # Beam Search
    beam_cons = beam_predict(hidden['GR'].values, tw_tvt, tw_gr, last_known_tvt, 10, 20.0, 144.0, 2)
    beam_loose = beam_predict(hidden['GR'].values, tw_tvt, tw_gr, last_known_tvt, 10, 8.0, 64.0, 2)

    features = pd.DataFrame({
        'well': well,
        'prediction_id': [f'{well}_{idx}' for idx in hidden.index],
        'last_known_tvt': np.float32(last_known_tvt),
        'frac_hidden': ((hidden.index - mask_start) / max(len(hidden) - 1, 1)).astype(np.float32),
        'md': hidden['MD'].values.astype(np.float32),
        'z': hidden['Z'].values.astype(np.float32),
        'x': hidden['X'].values.astype(np.float32),
        'y': hidden['Y'].values.astype(np.float32),
        'gr': gr_full.iloc[mask_start:].values,
        'gr_roll21': gr_full.rolling(21, center=True, min_periods=1).mean().iloc[mask_start:].values,
        'gr_std21': gr_full.rolling(21, center=True, min_periods=1).std().iloc[mask_start:].values,
        'dmd': (hidden['MD'] - float(last_known['MD'])).values.astype(np.float32),
        'dz': (hidden['Z'] - float(last_known['Z'])).values.astype(np.float32),
        'dist_xyz': np.sqrt((hidden['X']-last_known['X'])**2 + (hidden['Y']-last_known['Y'])**2 + (hidden['Z']-last_known['Z'])**2).values,
        'beam_cons_delta': (beam_cons - last_known_tvt).astype(np.float32),
        'beam_loose_delta': (beam_loose - last_known_tvt).astype(np.float32),
        'prefix_tvt_slope': np.float32(recent_slope(known['TVT_input'].values, known['MD'].values, 100))
    })

    for off in [-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80]:
        ref_gr = np.interp(last_known_tvt + off, tw_tvt, tw_gr)
        features[f'tw_diff_{int(off)}'] = features['gr'] - np.float32(ref_gr)

    if is_train:
        features['target'] = (hidden['TVT'].values - last_known_tvt).astype(np.float32)
    return features

## 3. Build, Train & Fix Leakage

In [ ]:
train_df = pd.concat([f for f in [build_hidden_features(f, DATA_PATH/'train'/f"{f.name.split('__')[0]}__typewell.csv", True) 
                                 for f in tqdm(sorted((DATA_PATH/'train').glob('*__horizontal_well.csv')))] if f is not None])
test_df = pd.concat([f for f in [build_hidden_features(f, DATA_PATH/'test'/f"{f.name.split('__')[0]}__typewell.csv", False) 
                                for f in tqdm(sorted((DATA_PATH/'test').glob('*__horizontal_well.csv')))] if f is not None])

# ВИКЛЮЧАЄМО 'beam_cons_delta' та 'beam_loose_delta' з навчання, щоб прибрати витік!
feats = [c for c in train_df.columns if c not in ['well', 'prediction_id', 'target', 'beam_cons_delta', 'beam_loose_delta']]

lgb_params = {
    "boosting_type": "gbdt", "learning_rate": 0.0599, "num_leaves": 89, "n_estimators": 5000,
    "reg_alpha": 2.03, "reg_lambda": 87.27, "colsample_bytree": 0.82, "subsample": 0.64,
    "device": "gpu", "verbose": -1
}

gkf = GroupKFold(n_splits=5)
test_df['pred_drift'] = 0.0

for fold, (tr_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df['well'])):
    X_tr, y_tr = train_df.iloc[tr_idx][feats], train_df.iloc[tr_idx]['target']
    X_va, y_va = train_df.iloc[val_idx][feats], train_df.iloc[val_idx]['target']
    model = LGBMRegressor(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[early_stopping(125), log_evaluation(500)])
    test_df['pred_drift'] += model.predict(test_df[feats]) / 5.0

test_df['tvt'] = test_df['last_known_tvt'] + test_df['pred_drift']
print(f"Унікальних значень TVT: {test_df['tvt'].nunique()}")

sample_sub = pd.read_csv(DATA_PATH / 'sample_submission.csv')
submission = sample_sub[['id']].merge(test_df.rename(columns={'prediction_id': 'id'})[['id', 'tvt']], on='id', how='left').ffill()
submission.to_csv("submission.csv", index=False)